# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides an end-to-end walkthrough for loading and exploring the FAIR² Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json).

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL for the FAIR² dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Instantiate the dataset
dataset = mlc.Dataset(croissant_url)
# Access the metadata object
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, their fields and columns, referencing everything by `@id` as required.

In [ ]:
# List all RecordSets and their corresponding fields by `@id`.
record_sets = metadata.record_sets

if not record_sets:
    print("No RecordSets found in the Croissant schema.")
else:
    print(f"Found {len(record_sets)} record set(s):")
    for rs in record_sets:
        print(f"- RecordSet @id: {rs['@id']}")
        rs_fields = rs.get('field', [])
        if isinstance(rs_fields, dict):
            rs_fields = [rs_fields]
        if rs_fields:
            print("  Fields in this RecordSet:")
            for field in rs_fields:
                if isinstance(field, dict):
                    print(f"    - {field['@id']}")
                else:
                    print(f"    - {field}")
        else:
            print("  No fields listed.")
        print("")

In [ ]:
# List a sample of the records from each RecordSet, referencing by `@id`.
if not record_sets:
    print("No records to display.")
else:
    for rs in record_sets:
        record_set_id = rs['@id']
        print(f"\nSample records for RecordSet @id: {record_set_id}")
        try:
            for i, record in enumerate(dataset.records(record_set=record_set_id)):
                print(record)
                if i >= 2:
                    break
        except Exception as e:
            print(f"  Could not load records: {e}")

## 3. Data Extraction
Load data from each record set into a pandas DataFrame for downstream analysis. Reference record sets and fields by their `@id` as above.

In [ ]:
# Create a DataFrame for each RecordSet, using @id for all references
dataframes = {}
record_set_ids = [rs['@id'] for rs in record_sets]
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"RecordSet {record_set_id}: Loaded {len(df)} records with columns:\n{df.columns.tolist()}")
    except Exception as e:
        print(f"  Could not load DataFrame for {record_set_id}: {e}")

# Show the head of the first record set as example
if record_set_ids:
    example_rs_id = record_set_ids[0]
    display(dataframes[example_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
We'll process one of the record sets (the main tabular dataset) by selecting a numeric field (column), removing outliers, normalizing the field, and then grouping the data by a categorical attribute. All columns are referenced strictly by their `@id` as found above.

In [ ]:
# Select the main tabular RecordSet for EDA (replace with the actual main @id if the dataset has only one)
main_record_set_id = record_set_ids[0] if record_set_ids else None

if main_record_set_id is not None:
    df = dataframes[main_record_set_id]
    print(f"Columns in record set {main_record_set_id}:\n{df.columns.tolist()}")
    # Try to find a likely numeric field (e.g. Age, tumor_size, interval, etc.)
    # Here we heuristically search for a numeric-looking column.
    numeric_field = None
    for col in df.columns:
        if df[col].dtype in ['int64', 'float64']:
            numeric_field = col
            break
    if numeric_field is None:
        # Try by heuristic matching
        for col in df.columns:
            if 'age' in col.lower() or 'interval' in col.lower() or 'size' in col.lower():
                numeric_field = col
                break
    print(f"Using numeric field for EDA: {numeric_field}")

    # Sample threshold; you may want to adjust based on field meaning
    threshold = 40
    if numeric_field:
        # Remove outliers (keep only records where value is above threshold)
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records where {numeric_field} > {threshold}:")
        print(filtered_df[[numeric_field]].head())

        # Normalize the selected numeric field
        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized values for {numeric_field}:")
        print(filtered_df[[numeric_field, norm_col]].head())

        # Group by a categorical column (e.g. sex, anatomical_site, msi_status)
        group_field = None
        for candidate in df.columns:
            if candidate != numeric_field and (df[candidate].dtype == 'O' or df[candidate].dtype.name == 'category'):
                group_field = candidate
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame(name=f"mean_{numeric_field}")
            print(f"\nGrouped mean {numeric_field} by {group_field}:")
            print(grouped_df.head())
        else:
            print("No suitable categorical field found for grouping.")
    else:
        print("No numeric field found for EDA.")
else:
    print("No record set available for EDA.")

## 5. Visualization
Let's visualize the distribution of the selected numeric field, and the grouped means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_field:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Visualize grouped mean if available
    if 'grouped_df' in locals() and group_field:
        plt.figure(figsize=(8,4))
        grouped_df.plot(kind='bar', legend=False)
        plt.title(f"Mean {numeric_field} grouped by {group_field}")
        plt.ylabel(f"Mean {numeric_field}")
        plt.xlabel(group_field)
        plt.tight_layout()
        plt.show()

## 6. Conclusion

In this notebook, we loaded and explored the FAIR² colorectal cancer dataset using its Croissant schema and the `mlcroissant` library. We demonstrated how to:
- Load metadata and derive the dataset structure referencing all entities by their `@id`.
- Enumerate available record sets, fields, and columns via their `@id`.
- Load actual records into pandas DataFrames.
- Perform common EDA, including filtering, normalization, grouping, and visualization, referencing all columns by `@id`.

This workflow provides a reproducible, standards-based approach for analyzing FAIR datasets with Python. For your dataset, adjust the field and record set IDs as shown above, and extend EDA as needed based on the research question.